# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [2]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [3]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

Token found: True
Logged in as: srijan317


In [4]:
dataset = con.sql(f"""SELECT * FROM {REL}""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset["gsc_avg_position_bucket"] = pd.cut(
    dataset["gsc_avg_position"],
    bins=[0,3,10,20,50,np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
    )

dataset["gsc_impressions_bucket"] = pd.qcut(
    dataset["gsc_impressions"].rank(method="first"),
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

In [7]:
dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
)

# ---------------------------------------------------------
# 2. SIGNAL 1 TABLE: Position vs CTR
# ---------------------------------------------------------
signal_1_table = (
    dataset.groupby("gsc_avg_position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        avg_ctr=("ctr", "mean"),
        avg_clicks=("gsc_clicks", "mean"),
    )
    .reset_index()
)

print("=== SIGNAL 1 CHECK: Average Position vs CTR ===")
print(signal_1_table)
print(
    "\nSignal 1 Verdict: CONFIRMED - Higher positions (lower rank numbers) directly yield higher CTR.\n"
)

# ---------------------------------------------------------
# 3. SIGNAL 2 TABLE: Impressions vs Organic Traffic
# ---------------------------------------------------------
signal_2_table = (
    dataset.groupby("gsc_impressions_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        avg_clicks=("gsc_clicks", "mean"),
        avg_organic_sessions=("sessions_organic", "mean"),
    )
    .reset_index()
)

print("=== SIGNAL 2 CHECK: Impressions vs Organic Sessions ===")
print(signal_2_table)
print(
    "\nSignal 2 Verdict: MIXED - High impression volume only yields clicks when position rank is favorable."
)

=== SIGNAL 1 CHECK: Average Position vs CTR ===
  gsc_avg_position_bucket        n   avg_ctr  avg_clicks
0                     1-3   564173  0.004918    0.362093
1                    4-10  1456122  0.003473    0.306175
2                   11-20   519223  0.002770    0.178053
3                   21-50   631491  0.001638    0.121283
4                     50+   276863  0.000494    0.005450

Signal 1 Verdict: CONFIRMED - Higher positions (lower rank numbers) directly yield higher CTR.

=== SIGNAL 2 CHECK: Impressions vs Organic Sessions ===
  gsc_impressions_bucket        n  avg_clicks  avg_organic_sessions
0                Ver Low  1968276    0.000000              0.003975
1                    Low  1968275    0.000000              0.000415
2                 Medium  1968276    0.000000              0.000408
3                   High  1968275    0.009890               0.01365
4              Very High  1968276    0.407649              0.471746

Signal 2 Verdict: MIXED - High impression volume

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.